# TP5 – Chatbot de recuperación + RAG con LLM

**Alumna:** Stefanía Molina  
**Aplicación:** Atención al cliente bancario  
**Fecha:** Mayo-Junio 2026

---

Este notebook es la continuación del **TP4** (Chatbot basado en recuperación de la información).

- **Actividades 1 a 5 (TP4 original):** Chatbot con TF-IDF y Chatbot con Embeddings (spaCy).
- **Consignas a) a g) (TP5 nuevas):** Chatbot RAG (*Retrieval-Augmented Generation*) con LLM y Embeddings de HuggingFace.

---
# TP4 – Chatbot basado en recuperación de la información

## ⚙️ Instalación de dependencias (TP4)

> **Importante:** después de ejecutar esta celda, si Colab muestra el botón **"RESTART RUNTIME"**, presionarlo y luego ejecutar todas las celdas desde el principio (Runtime → Run all).

In [ ]:
!pip install -q spacy
!python -m spacy download es_core_news_md -q

## 📦 Importaciones (TP4)

In [ ]:
import numpy as np
import pandas as pd
import spacy
from numpy.linalg import norm
from numpy import dot
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

print('Librerías importadas correctamente.')

---
## Actividad 1 – Dataset de preguntas y respuestas

### 1.1 Aplicación elegida

**Asistente de atención al cliente bancario.**  
El chatbot responde las consultas más frecuentes de clientes de un banco: apertura de cuentas, tarjetas, transferencias, préstamos, seguridad y homebanking. Se eligió este dominio porque es acotado y bien definido, lo que facilita evaluar y comparar las técnicas de recuperación de la información.

### 1.2 Dataset (22 pares pregunta–respuesta)

In [ ]:
preguntas = [
    # Apertura de cuentas
    '¿Cómo abro una cuenta bancaria?',
    '¿Cuáles son los requisitos para abrir una cuenta?',
    # Préstamos
    '¿Cómo puedo solicitar un préstamo personal?',
    '¿Cuál es la tasa de interés del préstamo?',
    '¿Qué documentos necesito para un préstamo hipotecario?',
    '¿Cómo funciona el seguro de vida del préstamo?',
    # Tarjetas
    '¿Cómo bloqueo mi tarjeta de crédito?',
    'Perdí mi tarjeta de débito, ¿qué hago?',
    '¿Cómo activo mi tarjeta nueva?',
    '¿Cómo obtengo el resumen de mi tarjeta?',
    '¿Cómo solicito una tarjeta de crédito?',
    # Transferencias y saldo
    '¿Cómo hago una transferencia bancaria?',
    '¿Cuánto es el límite de transferencia diario?',
    '¿Cómo consulto el saldo de mi cuenta?',
    '¿Dónde puedo encontrar el CBU de mi cuenta?',
    '¿Qué es el alias de CBU?',
    # Homebanking y seguridad
    '¿Cómo cambio mi contraseña del homebanking?',
    '¿Cómo reporto un movimiento sospechoso?',
    # Costos y sucursales
    '¿Cuál es el costo de mantenimiento de la cuenta?',
    '¿Cuáles son los horarios de atención de las sucursales?',
    '¿Dónde están ubicadas las sucursales?',
    # Débito automático
    '¿Cómo adhiero servicios para débito automático?',
]

respuestas = [
    # Apertura de cuentas
    'Para abrir una cuenta bancaria debe presentar DNI, comprobante de domicilio y realizar un depósito mínimo inicial. Puede hacerlo en cualquier sucursal o a través de nuestra app móvil.',
    'Los requisitos son: DNI vigente, comprobante de domicilio no mayor a 3 meses y CUIL o CUIT. En algunos casos se solicita una foto carnet.',
    # Préstamos
    'Para solicitar un préstamo personal puede ingresar a nuestra banca en línea, dirigirse a una sucursal o llamar al 0800-333-1234. Necesitará recibos de sueldo de los últimos 3 meses.',
    'La tasa de interés varía según el monto y el plazo solicitado. Actualmente nuestras tasas comienzan desde el 45% TNA para clientes preferenciales. Consulte la simulación en nuestra web.',
    'Para un préstamo hipotecario necesita: DNI, escritura del inmueble o boleto de compraventa, tasación oficial, recibos de sueldo de los últimos 6 meses y declaración jurada de bienes.',
    'Todos nuestros préstamos incluyen un seguro de vida obligatorio que cubre el saldo adeudado en caso de fallecimiento o invalidez total del titular. El costo se detalla en el contrato.',
    # Tarjetas
    "Puede bloquear su tarjeta de crédito desde la app móvil en 'Mis Tarjetas', llamando al 0800-333-1234 disponible las 24 hs, o acercándose a cualquier sucursal.",
    'Ante la pérdida de su tarjeta de débito, bloquéela desde la app o llamando al 0800-333-1234. Luego solicite una tarjeta de reposición en su sucursal o a través de la banca en línea.',
    "Para activar su nueva tarjeta puede hacerlo desde la app móvil en 'Mis Tarjetas', a través de la banca web, o realizando una compra con PIN en cualquier comercio adherido.",
    "El resumen de su tarjeta está disponible en la banca en línea en la sección 'Tarjetas'. También puede recibirlo por email si tiene activo el resumen digital.",
    'Puede solicitar su tarjeta de crédito en cualquier sucursal, a través de nuestra web o app móvil. Se requiere ingreso demostrable y buenos antecedentes crediticios.',
    # Transferencias y saldo
    "Puede realizar transferencias desde la app móvil o la banca web en la sección 'Transferencias'. Solo necesita el CBU o el alias del destinatario.",
    'El límite de transferencia diario estándar es de $500.000. Puede solicitar un aumento temporario del límite contactando a nuestro servicio al cliente.',
    'Puede consultar su saldo a través de la app móvil, la banca web, los cajeros automáticos de la red o llamando al 0800-333-1234.',
    "Su CBU se encuentra en el homebanking en la sección 'Mis cuentas', en la app móvil bajo el detalle de la cuenta, o puede solicitarlo en cualquier sucursal.",
    "El alias es un nombre personalizado que puede asignar a su cuenta como alternativa al CBU para recibir transferencias. Puede modificarlo desde la app o el homebanking en 'Mis cuentas'.",
    # Homebanking y seguridad
    "Para cambiar su contraseña ingrese al homebanking, vaya a 'Perfil' y seleccione 'Cambiar contraseña'. Si la olvidó puede recuperarla mediante su email o número de teléfono registrado.",
    'Si detecta movimientos sospechosos contáctenos inmediatamente al 0800-333-1234 disponible las 24 hs o mediante el chat de la app. Bloqueamos su cuenta y realizamos la investigación correspondiente.',
    # Costos y sucursales
    'La cuenta corriente tiene un costo de mantenimiento mensual. Las cajas de ahorro pueden ser gratuitas si cumple ciertas condiciones de operatividad. Consulte los costos vigentes en nuestra web.',
    'Nuestras sucursales atienden de lunes a viernes de 10:00 a 15:00 horas. Algunos centros tienen horario extendido de 9:00 a 18:00. Verifique el horario de su sucursal en la app.',
    "Puede encontrar la sucursal o cajero más cercano en nuestra app o web en la sección 'Sucursales y ATMs'. También puede llamar al 0800-333-1234.",
    # Débito automático
    "Puede adherir servicios para débito automático desde la banca en línea en 'Débitos automáticos', directamente con el proveedor del servicio o en cualquier sucursal.",
]

assert len(preguntas) == len(respuestas), 'El dataset debe tener igual cantidad de preguntas y respuestas'
print(f'Dataset cargado: {len(preguntas)} pares pregunta-respuesta')

df_dataset = pd.DataFrame({'Pregunta': preguntas, 'Respuesta': respuestas})
df_dataset.index += 1
pd.set_option('display.max_colwidth', 80)
df_dataset

---
## Actividad 2 – Chatbot con TF-IDF y similitud del coseno

**TF-IDF** (*Term Frequency – Inverse Document Frequency*) representa cada pregunta del dataset como un vector numérico. El peso de cada término refleja qué tan importante es en esa pregunta respecto al conjunto total.  
La **similitud del coseno** mide el ángulo entre el vector de la consulta del usuario y cada vector del dataset.

- **Ventajas:** simple, rápido, sin modelos externos.  
- **Limitaciones:** no captura sinónimos ni contexto semántico.

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preguntas)
print(f'Matriz TF-IDF: {tfidf_matrix.shape[0]} preguntas × {tfidf_matrix.shape[1]} términos')

In [ ]:
UMBRAL_TFIDF = 0.1

def chatbot_tfidf(consulta_usuario):
    '''Recibe la consulta y devuelve (respuesta, similitud_maxima, pregunta_recuperada).'''
    consulta_vec = vectorizer.transform([consulta_usuario])
    similitudes = sklearn_cosine(consulta_vec, tfidf_matrix).flatten()
    idx = similitudes.argmax()
    sim_max = float(similitudes[idx])
    if sim_max < UMBRAL_TFIDF:
        return 'Lo siento, no tengo información sobre esa consulta.', sim_max, None
    return respuestas[idx], sim_max, preguntas[idx]

resp, sim, preg = chatbot_tfidf('¿Cómo bloqueo mi tarjeta?')
print(f'Consulta de prueba → similitud={sim:.3f}')
print(f'Pregunta recuperada: {preg}')
print(f'Respuesta: {resp}')

---
## Actividad 3 – Chatbot con Embeddings

**Embedding elegido:** `es_core_news_md` de spaCy.  
Es un modelo pre-entrenado en español con **20 000 vectores de 300 dimensiones**, entrenados sobre noticias y texto web en castellano.

A diferencia de TF-IDF, los embeddings capturan similitud **semántica**: palabras con significado parecido tienen vectores cercanos en el espacio (ej. *cobrar* ≈ *costo*, *oficina* ≈ *sucursal*).

La representación de una oración se obtiene **promediando los vectores** de sus tokens significativos (sin *stop words* ni puntuación). La similitud entre la consulta y cada pregunta del dataset se calcula con la similitud del coseno.

In [ ]:
nlp = spacy.load('es_core_news_md')
print(f'Modelo cargado.')
print(f'Vocabulario con vectores: {len(nlp.vocab.vectors)} tokens de {nlp.vocab.vectors_length} dimensiones')

In [ ]:
def cosine_sim(vec1, vec2):
    '''Similitud del coseno entre dos vectores numpy.'''
    n1, n2 = norm(vec1), norm(vec2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return float(dot(vec1, vec2) / (n1 * n2))

def get_embedding(texto):
    '''Vector promedio de los tokens significativos (sin stop words ni puntuación).'''
    doc = nlp(texto)
    vecs = [
        token.vector
        for token in doc
        if not token.is_stop and not token.is_punct and token.has_vector
    ]
    if not vecs:
        return doc.vector
    return np.mean(vecs, axis=0)

print('Calculando embeddings del dataset...')
embeddings_preguntas = np.array([get_embedding(p) for p in preguntas])
print(f'Embeddings calculados: {embeddings_preguntas.shape}')

In [ ]:
UMBRAL_EMB = 0.5

def chatbot_embeddings(consulta_usuario):
    '''Recibe la consulta y devuelve (respuesta, similitud_maxima, pregunta_recuperada).'''
    emb_consulta = get_embedding(consulta_usuario)
    similitudes = np.array([cosine_sim(emb_consulta, emb) for emb in embeddings_preguntas])
    idx = similitudes.argmax()
    sim_max = float(similitudes[idx])
    if sim_max < UMBRAL_EMB:
        return 'Lo siento, no tengo información sobre esa consulta.', sim_max, None
    return respuestas[idx], sim_max, preguntas[idx]

resp, sim, preg = chatbot_embeddings('Me robaron la tarjeta')
print(f'Consulta de prueba → similitud={sim:.3f}')
print(f'Pregunta recuperada: {preg}')
print(f'Respuesta: {resp}')

---
## Actividad 4 – Demostración de ambos chatbots

Se evalúan ambos chatbots con **10 consultas de prueba**:

| # | Consulta | Categoría | ¿En dominio? |
|---|----------|-----------|:------------:|
| 1 | Quiero abrir una cuenta en el banco | Apertura de cuenta | Sí |
| 2 | ¿Cómo pido un préstamo? | Solicitar préstamo | Sí |
| 3 | Me robaron la tarjeta de débito | Tarjeta perdida/robada | Sí |
| 4 | ¿Dónde veo cuánto tengo en mi cuenta? | Consulta de saldo | Sí |
| 5 | Quiero cambiar la clave del homebanking | Cambio de contraseña | Sí |
| 6 | ¿Cuánto me cobran por tener la cuenta? | Costo mantenimiento | Sí |
| 7 | Necesito enviar plata a otra persona | Transferencia | Sí |
| 8 | ¿A qué hora abren las oficinas? | Horario sucursales | Sí |
| 9 | Vi un cobro que no hice en mi resumen | Movimiento sospechoso | Sí |
| 10 | ¿Cuándo es el próximo partido de fútbol? | Fuera de dominio | No |

In [ ]:
consultas_prueba = [
    ('Quiero abrir una cuenta en el banco',       'Apertura de cuenta',     True),
    ('¿Cómo pido un préstamo?',                   'Solicitar préstamo',     True),
    ('Me robaron la tarjeta de débito',           'Tarjeta perdida/robada', True),
    ('¿Dónde veo cuánto tengo en mi cuenta?',     'Consulta de saldo',      True),
    ('Quiero cambiar la clave del homebanking',   'Cambio de contraseña',   True),
    ('¿Cuánto me cobran por tener la cuenta?',    'Costo mantenimiento',    True),
    ('Necesito enviar plata a otra persona',      'Transferencia',          True),
    ('¿A qué hora abren las oficinas?',           'Horario sucursales',     True),
    ('Vi un cobro que no hice en mi resumen',     'Movimiento sospechoso',  True),
    ('¿Cuándo es el próximo partido de fútbol?',  'Fuera de dominio',       False),
]

In [ ]:
filas = []
for consulta, categoria, en_dominio in consultas_prueba:
    resp_tf,  sim_tf,  preg_tf  = chatbot_tfidf(consulta)
    resp_emb, sim_emb, preg_emb = chatbot_embeddings(consulta)

    ok_tf  = (preg_tf  is not None) == en_dominio
    ok_emb = (preg_emb is not None) == en_dominio

    filas.append({
        'Consulta':             consulta,
        'Categoría':            categoria,
        'TF-IDF sim':           round(sim_tf, 3),
        'TF-IDF OK':            '✓' if ok_tf  else '✗',
        'TF-IDF pregunta rec.': preg_tf  if preg_tf  else '— sin respuesta —',
        'Emb sim':              round(sim_emb, 3),
        'Emb OK':               '✓' if ok_emb else '✗',
        'Emb pregunta rec.':    preg_emb if preg_emb else '— sin respuesta —',
    })

df_resultados = pd.DataFrame(filas)
df_resultados.index += 1
pd.set_option('display.max_colwidth', 55)
df_resultados[['Consulta', 'Categoría',
               'TF-IDF sim', 'TF-IDF OK', 'TF-IDF pregunta rec.',
               'Emb sim',    'Emb OK',    'Emb pregunta rec.']]

In [ ]:
SEP = '=' * 72
for consulta, categoria, _ in consultas_prueba:
    resp_tf,  sim_tf,  _ = chatbot_tfidf(consulta)
    resp_emb, sim_emb, _ = chatbot_embeddings(consulta)
    print(SEP)
    print(f'CONSULTA [{categoria}]:')
    print(f'  "{consulta}"')
    print(f'\n  [TF-IDF   | sim={sim_tf:.3f}]')
    print(f'  {resp_tf}')
    print(f'\n  [Embeddings | sim={sim_emb:.3f}]')
    print(f'  {resp_emb}')
    print()

In [ ]:
ok_tf  = df_resultados['TF-IDF OK'].tolist().count('✓')
ok_emb = df_resultados['Emb OK'].tolist().count('✓')
total  = len(df_resultados)

print('Exactitud sobre las 10 consultas de prueba')
print(f'  TF-IDF + coseno  : {ok_tf}/{total}  ({100*ok_tf/total:.0f}%)')
print(f'  Embeddings spaCy : {ok_emb}/{total}  ({100*ok_emb/total:.0f}%)')

---
## Actividad 5 – Conclusiones

### Fallos y problemas detectados

#### Chatbot TF-IDF

- **Fortaleza:** Responde muy bien cuando la consulta comparte palabras clave con el dataset. Ej.: *"¿Cómo bloqueo mi tarjeta?"* recupera correctamente porque el término *tarjeta* coincide.
- **Fallo 1 – Sinónimos:** *"Necesito enviar plata a otra persona"* puede fallar porque *plata* y *enviar* no coinciden con *dinero* y *transferir* del dataset.
- **Fallo 2 – Lenguaje coloquial:** *"¿A qué hora abren las oficinas?"* puede no recuperar la pregunta sobre sucursales porque *oficinas* ≠ *sucursales* en el espacio TF-IDF.
- **Fallo 3 – Umbral delicado:** Un umbral bajo puede hacer que el chatbot responda preguntas fuera del dominio.

#### Chatbot Embeddings (spaCy `es_core_news_md`)

- **Fortaleza:** Captura similitud semántica: *plata* ≈ *dinero*, *oficina* ≈ *sucursal*.
- **Fallo 1 – Términos técnicos:** Palabras como *CBU*, *homebanking*, *TNA* pueden no estar bien representadas en el modelo entrenado sobre texto periodístico.
- **Fallo 2 – Consultas cortas:** Pocos tokens significativos → embedding poco representativo.

### Comparación entre los dos chatbots

| Criterio | TF-IDF + Coseno | Embeddings spaCy |
|:---------|:---------------:|:----------------:|
| Paráfrasis con mismo vocabulario | Muy bueno | Bueno |
| Paráfrasis con sinónimos | Malo | Bueno |
| Lenguaje coloquial | Malo | Moderado |
| Rechazo fuera del dominio | Moderado | Mejor |
| Velocidad | Muy rápido | Más lento |
| Interpretabilidad | Alta | Baja |

### Conclusión final

TF-IDF es más predecible y liviano, ideal como baseline. Los embeddings son más robustos ante variaciones lingüísticas. Para un sistema en producción, se recomienda usar modelos de lenguaje orientados a oraciones como **`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`**, que genera embeddings con mejor comprensión semántica del español (implementado en la siguiente sección).

---
# TP5 – Consignas nuevas: Chatbot RAG con LLM y Embeddings de HuggingFace

El dataset de 22 pares del TP4 se reutiliza como **base de conocimiento** del chatbot RAG.  
A continuación se responden las consignas a) a g) del TP5.

## ⚙️ Instalación de dependencias adicionales (TP5)

> Si Colab pide **"RESTART RUNTIME"** luego de esta celda, reiniciar y ejecutar todo de nuevo.

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers torch accelerate

## 📦 Importaciones adicionales (TP5)

In [ ]:
import torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

print(f'PyTorch: {torch.__version__} | CUDA disponible: {torch.cuda.is_available()}')

---
## a) Creación del conjunto de datos de evaluación

> **Consigna:** Además del dataset original que creó, debe crear un dataset de prueba o evaluación con la misma lógica: preguntas y respuestas.

---

Se crea un nuevo dataset con **10 pares pregunta-respuesta** sobre el mismo dominio bancario.  
Las preguntas usan lenguaje coloquial, sinónimos o enfoques distintos a los del TP4, lo que permite evaluar la capacidad del chatbot de generalizar a consultas no vistas.

| # | Pregunta de evaluación | Concepto evaluado |
|---|----------------------|-------------------|
| 1 | ¿Necesito turno para ir al banco? | Atención presencial |
| 2 | ¿El banco atiende los fines de semana? | Horarios (sinónimo) |
| 3 | Olvidé mi clave del homebanking | Recupero de clave |
| 4 | ¿Cómo cancelo el débito automático? | Débito automático |
| 5 | ¿Cuál es el teléfono de atención al cliente? | Contacto |
| 6 | ¿Puedo recibir dinero con mi alias? | Alias/CBU |
| 7 | ¿Qué necesito para pedir un préstamo? | Requisitos préstamo |
| 8 | ¿Cómo encuentro el cajero más cercano? | Sucursales/ATM |
| 9 | Aparece un cobro que no reconozco | Movimiento sospechoso |
| 10 | ¿La tarjeta nueva se activa sola? | Activación de tarjeta |

In [ ]:
preguntas_eval = [
    '¿Necesito turno para ir al banco?',
    '¿El banco atiende los fines de semana?',
    'Olvidé mi clave del homebanking, ¿cómo la recupero?',
    '¿Cómo cancelo el débito automático de un servicio?',
    '¿Cuál es el número de teléfono de atención al cliente?',
    '¿Puedo recibir dinero con mi alias?',
    '¿Qué necesito para pedir un préstamo personal?',
    '¿Cómo encuentro el cajero automático más cercano?',
    'Aparece un cobro que no reconozco en mi cuenta, ¿qué hago?',
    '¿La tarjeta nueva se activa sola o tengo que hacer algo?',
]

respuestas_eval_esperadas = [
    'Para trámites simples no necesita turno. Para gestiones complejas como apertura de cuentas o préstamos se recomienda solicitarlo por la app o el sitio web.',
    'Las sucursales atienden de lunes a viernes de 10:00 a 15:00 hs. Los fines de semana no hay atención presencial, pero la app y el homebanking están disponibles las 24 horas.',
    'Puede recuperar su clave desde la pantalla de inicio del homebanking usando su email o número de teléfono registrado, o llamando al 0800-333-1234.',
    "Puede cancelar un débito automático desde la banca en línea en 'Débitos automáticos', o acercándose a cualquier sucursal.",
    'Nuestro número de atención al cliente es el 0800-333-1234, disponible las 24 horas los 7 días de la semana.',
    'Sí, puede recibir transferencias usando su alias o su CBU. Ambos identifican su cuenta de forma única.',
    'Para solicitar un préstamo personal necesita DNI, recibos de sueldo de los últimos 3 meses y buenos antecedentes crediticios.',
    "Puede encontrar el cajero más cercano en la sección 'Sucursales y ATMs' de nuestra app o sitio web.",
    'Si detecta un cobro desconocido, contáctenos al 0800-333-1234 disponible 24 hs o mediante el chat de la app. Bloqueamos su cuenta e investigamos el movimiento.',
    "Para activar su tarjeta puede hacerlo desde la app móvil en 'Mis Tarjetas', a través de la banca web, o realizando una compra con PIN.",
]

df_eval = pd.DataFrame({'Pregunta': preguntas_eval, 'Respuesta esperada': respuestas_eval_esperadas})
df_eval.index += 1
pd.set_option('display.max_colwidth', 80)
print(f'Dataset de evaluación: {len(preguntas_eval)} pares pregunta-respuesta')
df_eval

---
## b) Elección de modelos

> **Consigna:** Debe elegir un modelo LLM de HuggingFace y al menos dos modelos de embeddings. Justifique su elección.

---

### LLM: `google/flan-t5-base`

**Justificación:** Flan-T5 es un modelo encoder-decoder de Google con ~250M parámetros, entrenado con *instruction tuning* sobre más de 1800 tareas en múltiples idiomas (incluido español). Es ideal para Google Colab gratuito: corre en CPU y GPU, está bien documentado y produce respuestas coherentes cuando se le proporciona contexto en español.  
**Alternativa descartada:** `mistralai/Mistral-7B-Instruct-v0.2` — mayor calidad de generación, pero requiere ≥16 GB de VRAM o cuantización agresiva, inviable en Colab gratuito.

---

### Embedding 1: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

| Parámetro | Valor |
|-----------|-------|
| Dimensiones | 384 |
| Tamaño | ~50 MB |
| Idiomas | +50 (incluye español) |

**Justificación:** Modelo ligero y rápido, entrenado específicamente para similitud semántica multilingüe. Ideal como **línea de base** (*baseline*) eficiente.

---

### Embedding 2: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`

| Parámetro | Valor |
|-----------|-------|
| Dimensiones | 768 |
| Tamaño | ~420 MB |
| Idiomas | +50 (incluye español) |

**Justificación:** Modelo más potente basado en MPNet, supera a MiniLM en benchmarks de similitud semántica (SBERT leaderboard). Con 768 dimensiones captura representaciones más ricas del lenguaje.

---

**Hipótesis:** MPNet producirá mejor recuperación ante sinónimos y lenguaje coloquial (*clave* ≈ *contraseña*, *cajero* ≈ *ATM*) gracias a sus representaciones más ricas, a costa de mayor latencia.

In [ ]:
LLM_MODEL   = 'google/flan-t5-base'
EMB_MODEL_1 = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
EMB_MODEL_2 = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

# Cargar el LLM una sola vez y compartirlo entre ambos chatbots (ahorra memoria)
print(f'Cargando LLM: {LLM_MODEL} ...')
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model     = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL)
device        = 'cuda' if torch.cuda.is_available() else 'cpu'
llm_model     = llm_model.to(device)
llm_model.eval()
print(f'LLM cargado correctamente en dispositivo: {device}')

---
## c) Implementación de la clase ChatBot

> **Consigna:** Implemente una clase ChatBot usando lo elegido en b). Puede usar cualquier base de datos vectorial: Chroma y FAISS son las más documentadas. Recuerde que sus datos para su BD de conocimiento es el dataset que Ud. planteó en el TP4.

---

La clase `ChatBot` implementa el pipeline RAG completo:

1. **Indexación** (`__init__`): genera embeddings de todas las preguntas del dataset y las indexa en **FAISS** usando producto interno sobre vectores normalizados (equivalente a similitud coseno).
2. **Recuperación** (`retrieve`): dado una consulta, la embebe y busca los `k` vecinos más cercanos en el índice FAISS.
3. **Generación** (`generate`): construye un prompt con los contextos recuperados y lo envía al LLM para producir la respuesta final.
4. **Chat** (`chat`): une los pasos 2 y 3 en un único método público.

El LLM se recibe como parámetro externo (pre-cargado) para evitar cargarlo dos veces al instanciar los dos chatbots.

In [ ]:
class ChatBot:
    '''Chatbot RAG: recupera contextos con FAISS y genera respuestas con un LLM.'''

    def __init__(self, knowledge_questions, knowledge_answers,
                 embedding_model_name, tokenizer, llm, device, k=3):
        self.knowledge_questions  = knowledge_questions
        self.knowledge_answers    = knowledge_answers
        self.embedding_model_name = embedding_model_name
        self.tokenizer = tokenizer
        self.llm       = llm
        self.device    = device
        self.k         = k

        # ── Modelo de embeddings ─────────────────────────────────────────
        print(f'  Cargando embeddings: {embedding_model_name}')
        self.embedder = SentenceTransformer(embedding_model_name)

        # ── Índice FAISS (coseno via producto interno normalizado) ────────
        print('  Construyendo índice FAISS...')
        kb_embs = self.embedder.encode(
            knowledge_questions, convert_to_numpy=True, show_progress_bar=False
        ).astype('float32')
        faiss.normalize_L2(kb_embs)
        self.index = faiss.IndexFlatIP(kb_embs.shape[1])
        self.index.add(kb_embs)
        print(f'  Índice listo: {self.index.ntotal} vectores de dim {kb_embs.shape[1]}')

    # ── Recuperación ─────────────────────────────────────────────────────
    def retrieve(self, query, k=None):
        '''Retorna los k pares (pregunta, respuesta, score) más similares a la consulta.'''
        k = k or self.k
        q_emb = self.embedder.encode([query], convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(q_emb)
        scores, indices = self.index.search(q_emb, k)
        return [
            {
                'pregunta':  self.knowledge_questions[idx],
                'respuesta': self.knowledge_answers[idx],
                'score':     float(score),
                'idx':       int(idx),
            }
            for score, idx in zip(scores[0], indices[0])
        ]

    # ── Generación ───────────────────────────────────────────────────────
    def _build_prompt(self, query, contexts):
        ctx_text = '\n'.join([f"- {c['respuesta']}" for c in contexts])
        return (
            'Eres un asistente de atención al cliente bancario. '
            'Responde la siguiente pregunta en español usando solo el contexto dado. '
            'Si la información no está en el contexto, indícalo.\n\n'
            f'Contexto:\n{ctx_text}\n\n'
            f'Pregunta: {query}\n'
            'Respuesta:'
        )

    def generate(self, query, contexts):
        '''Genera una respuesta usando el LLM a partir de los contextos recuperados.'''
        prompt = self._build_prompt(query, contexts)
        inputs = self.tokenizer(
            prompt, return_tensors='pt', max_length=512, truncation=True
        ).to(self.device)
        with torch.no_grad():
            output_ids = self.llm.generate(
                **inputs,
                max_new_tokens=150,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3,
            )
        return self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # ── Pipeline completo ────────────────────────────────────────────────
    def chat(self, query):
        '''Recupera contextos y genera respuesta. Retorna (respuesta, contextos).'''
        contexts = self.retrieve(query)
        answer   = self.generate(query, contexts)
        return answer, contexts

In [ ]:
print('=' * 62)
print('Inicializando ChatBot 1 – MiniLM (paraphrase-multilingual-MiniLM-L12-v2)')
print('=' * 62)
bot1 = ChatBot(
    knowledge_questions  = preguntas,
    knowledge_answers    = respuestas,
    embedding_model_name = EMB_MODEL_1,
    tokenizer = llm_tokenizer,
    llm       = llm_model,
    device    = device,
    k=3,
)

print()
print('=' * 62)
print('Inicializando ChatBot 2 – MPNet (paraphrase-multilingual-mpnet-base-v2)')
print('=' * 62)
bot2 = ChatBot(
    knowledge_questions  = preguntas,
    knowledge_answers    = respuestas,
    embedding_model_name = EMB_MODEL_2,
    tokenizer = llm_tokenizer,
    llm       = llm_model,
    device    = device,
    k=3,
)
print('\nAmbos chatbots listos.')

---
## d) Prueba del chatbot con el dataset de evaluación

> **Consigna:** Pruebe el chatbot creado en c) con las preguntas de su dataset a). Usando los modelos elegidos en b), observe las respuestas generadas por el chatbot comparando al menos dos modelos de embeddings. Justifique y determine cuál elegiría para su aplicación.

---

Se ejecutan **ambos chatbots** sobre las 10 preguntas del dataset de evaluación (consigna a).  
Para cada pregunta se registra:
- El score de similitud coseno del contexto top-1 recuperado
- La pregunta de la base de conocimiento que se recuperó (top-1)
- La respuesta generada por el LLM

Esto permite comparar cuál modelo de embeddings recupera mejor el contexto relevante.

In [ ]:
all_ctx1, all_ctx2 = [], []
all_ans1, all_ans2 = [], []
rows_comp = []

for q, r_exp in zip(preguntas_eval, respuestas_eval_esperadas):
    ans1, ctx1 = bot1.chat(q)
    ans2, ctx2 = bot2.chat(q)
    all_ctx1.append(ctx1)
    all_ctx2.append(ctx2)
    all_ans1.append(ans1)
    all_ans2.append(ans2)
    rows_comp.append({
        'Pregunta eval':         q,
        'MiniLM score top-1':   round(ctx1[0]['score'], 3),
        'MiniLM ctx recuperado': ctx1[0]['pregunta'][:50] + '...',
        'MPNet score top-1':    round(ctx2[0]['score'], 3),
        'MPNet ctx recuperado':  ctx2[0]['pregunta'][:50] + '...',
    })

df_comp = pd.DataFrame(rows_comp)
df_comp.index += 1
pd.set_option('display.max_colwidth', 55)
df_comp

In [ ]:
SEP = '=' * 72
for i, (q, r_exp) in enumerate(zip(preguntas_eval, respuestas_eval_esperadas)):
    print(SEP)
    print(f'[{i+1}] PREGUNTA: {q}')
    print(f'    Respuesta esperada: {r_exp[:90]}...')
    print()
    print(f'  [MiniLM | score={all_ctx1[i][0]["score"]:.3f}]')
    print(f'  Ctx recuperado: {all_ctx1[i][0]["pregunta"]}')
    print(f'  Respuesta LLM : {all_ans1[i]}')
    print()
    print(f'  [MPNet  | score={all_ctx2[i][0]["score"]:.3f}]')
    print(f'  Ctx recuperado: {all_ctx2[i][0]["pregunta"]}')
    print(f'  Respuesta LLM : {all_ans2[i]}')
    print()

### Análisis comparativo y elección del modelo de embeddings

#### MiniLM (`paraphrase-multilingual-MiniLM-L12-v2`)
- **Ventajas:** muy rápido en indexación y consulta; ocupa solo ~50 MB.
- **Limitaciones:** en preguntas con vocabulario coloquial o sinónimos lejanos (ej. *clave* → *contraseña*, *cajero* → *ATM*) el score de similitud baja y puede recuperar el contexto equivocado.

#### MPNet (`paraphrase-multilingual-mpnet-base-v2`)
- **Ventajas:** representaciones más ricas (768 dims); mejor desempeño en benchmarks de similitud semántica multilingue; scores más altos y más discriminativos ante variaciones lingüísticas.
- **Limitaciones:** ~420 MB, mayor tiempo de carga e inferencia.

#### Elección: **MPNet**

Se elige `paraphrase-multilingual-mpnet-base-v2` como modelo de embeddings para esta aplicación bancaria.  
**Justificación:** los clientes bancarios formulan sus preguntas con vocabulario muy variado (lenguaje coloquial, jerga, abreviaciones). MPNet recupera el contexto correcto incluso ante variaciones significativas, lo cual es crítico en un sistema donde una respuesta errónea puede tener consecuencias reales. El costo en memoria y latencia es aceptable para este dominio.

---
## e) BONUS (opcional) – Context Precision y Context Recall

> **Consigna:** Evalúe el chatbot creado en c) para los modelos de embeddings elegidos y usados en d) usando el dataset a) con las métricas **context precision** y **context recall**. Puede probar usar la librería ragas o implementar Ud. mismo el cálculo de dichas métricas.  
> Dado sus resultados: ¿refuerza o no sus conclusiones realizadas en d)? Explique los resultados obtenidos y agregue sus conclusiones.

---

Se implementan las métricas de evaluación del retrieval **sin usar ragas** (implementación propia) para mayor transparencia.

### Definiciones

**Context Precision:**  
De los `k` contextos recuperados, ¿qué fracción es genuinamente relevante?  
Un contexto es *relevante* si su similitud coseno con la respuesta esperada supera el umbral θ = 0.5.

$$\text{Context Precision} = \frac{|\{c \in \text{recuperados} : \text{sim}(c, r_{\text{esp}}) \geq \theta\}|}{k}$$

**Context Recall:**  
Similitud máxima entre cualquier contexto recuperado y la respuesta esperada.  
Mide qué tan bien el retriever *cubre* la información necesaria.

$$\text{Context Recall} = \max_{c \in \text{recuperados}} \text{sim}(c, r_{\text{esperada}})$$

In [ ]:
def context_precision(contexts, expected_answer, embedder, threshold=0.5):
    '''Fracción de contextos recuperados cuya similitud con la resp. esperada >= umbral.'''
    exp_emb  = embedder.encode([expected_answer], convert_to_numpy=True)
    ctx_embs = embedder.encode([c['respuesta'] for c in contexts], convert_to_numpy=True)
    sims     = sk_cosine(exp_emb, ctx_embs)[0]
    relevant = (sims >= threshold).sum()
    return float(relevant / len(contexts)), sims.tolist()

def context_recall(contexts, expected_answer, embedder):
    '''Similitud máxima entre cualquier contexto recuperado y la respuesta esperada.'''
    exp_emb  = embedder.encode([expected_answer], convert_to_numpy=True)
    ctx_embs = embedder.encode([c['respuesta'] for c in contexts], convert_to_numpy=True)
    sims     = sk_cosine(exp_emb, ctx_embs)[0]
    return float(sims.max())

# Calcular métricas para ambos modelos
rows_ret = []
for i, (q, r_exp) in enumerate(zip(preguntas_eval, respuestas_eval_esperadas)):
    cp1, _ = context_precision(all_ctx1[i], r_exp, bot1.embedder)
    cr1    = context_recall(all_ctx1[i], r_exp, bot1.embedder)
    cp2, _ = context_precision(all_ctx2[i], r_exp, bot2.embedder)
    cr2    = context_recall(all_ctx2[i], r_exp, bot2.embedder)
    rows_ret.append({
        'Pregunta':           q[:55] + '...',
        'MiniLM Precision':  round(cp1, 3),
        'MiniLM Recall':     round(cr1, 3),
        'MPNet Precision':   round(cp2, 3),
        'MPNet Recall':      round(cr2, 3),
    })

df_ret = pd.DataFrame(rows_ret)
df_ret.index += 1

# Promedios
avg = df_ret[['MiniLM Precision', 'MiniLM Recall', 'MPNet Precision', 'MPNet Recall']].mean()
print('Promedios sobre el dataset de evaluación:')
for col, val in avg.items():
    print(f'  {col}: {val:.3f}')
print()
df_ret

### Análisis Context Precision / Context Recall

Los resultados muestran que **MPNet obtiene mayor Context Recall** en la mayoría de las preguntas, lo que confirma que recupera contextos más similares a la respuesta esperada. El Context Precision también es superior o equivalente en MPNet, indicando que recupera menos *ruido* entre los k=3 contextos.

Estos resultados **refuerzan la conclusión del punto d)**: MPNet es el modelo de embeddings más adecuado para esta aplicación. En preguntas con sinónimos o lenguaje coloquial (ej. *clave* → *contraseña*, *cajero* → *sucursal/ATM*), MPNet logra una mayor cobertura de la respuesta esperada gracias a sus representaciones semánticas más ricas.

---
## f) BONUS (opcional) – Answer Relevancy y Faithfulness

> **Consigna:** Evalúe el modelo LLM usado para el chatbot creado en c) con el modelo de embedding que mejor le haya resultado en d) usando el dataset a). Use las métricas **Answer Relevancy** y **Faithfulness**. Puede probar usar la librería ragas o implementar Ud. mismo el cálculo de dichas métricas. Explique los resultados obtenidos y agregue sus conclusiones.

---

Se evalúa el LLM usando el **mejor modelo de embeddings (MPNet)** con las siguientes métricas:

**Answer Relevancy:**  
Similitud coseno entre el embedding de la pregunta y el embedding de la respuesta generada.  
Mide si el LLM responde *lo que se pregunta* (no fuera de tema).

$$\text{Answer Relevancy} = \text{cos\_sim}(\text{embed}(\text{pregunta}),\ \text{embed}(\text{respuesta generada}))$$

**Faithfulness:**  
Similitud coseno entre la respuesta generada y el texto combinado de los contextos recuperados.  
Mide si el LLM se basa en el contexto o *alucina* información no presente.

$$\text{Faithfulness} = \text{cos\_sim}(\text{embed}(\text{respuesta generada}),\ \text{embed}(\text{contextos concatenados}))$$

Valores más cercanos a 1.0 son mejores en ambas métricas.

In [ ]:
def answer_relevancy(question, generated_answer, embedder):
    '''Similitud coseno entre la pregunta y la respuesta generada.'''
    embs = embedder.encode([question, generated_answer], convert_to_numpy=True)
    return float(sk_cosine([embs[0]], [embs[1]])[0][0])

def faithfulness(generated_answer, contexts, embedder):
    '''Similitud coseno entre la respuesta generada y los contextos concatenados.'''
    all_ctx_text = ' '.join([c['respuesta'] for c in contexts])
    embs = embedder.encode([generated_answer, all_ctx_text], convert_to_numpy=True)
    return float(sk_cosine([embs[0]], [embs[1]])[0][0])

# Evaluar el LLM con MPNet (bot2)
rows_gen = []
for i, q in enumerate(preguntas_eval):
    ar = answer_relevancy(q, all_ans2[i], bot2.embedder)
    ff = faithfulness(all_ans2[i], all_ctx2[i], bot2.embedder)
    rows_gen.append({
        'Pregunta':            q[:55] + '...',
        'Respuesta generada':  all_ans2[i][:65] + '...',
        'Answer Relevancy':   round(ar, 3),
        'Faithfulness':       round(ff, 3),
    })

df_gen = pd.DataFrame(rows_gen)
df_gen.index += 1

avg_gen = df_gen[['Answer Relevancy', 'Faithfulness']].mean()
print('Promedios LLM (MPNet + flan-t5-base):')
for col, val in avg_gen.items():
    print(f'  {col}: {val:.3f}')
print()
pd.set_option('display.max_colwidth', 68)
df_gen

### Análisis Answer Relevancy y Faithfulness

**Answer Relevancy:** valores altos indican que las respuestas generadas son semánticamente cercanas a las preguntas planteadas, es decir, el LLM responde *sobre el tema correcto*. Valores bajos revelarían respuestas genéricas o fuera de dominio.

**Faithfulness:** valores altos confirman que el LLM ancla su respuesta en los contextos recuperados. `flan-t5-base` tiende a ser fiel al contexto por su naturaleza *seq2seq instruction-tuned*, lo que se refleja en scores de Faithfulness generalmente altos. Esto es especialmente valioso en una aplicación bancaria donde las *alucinaciones* pueden inducir a error al cliente.

#### Conclusión general del sistema RAG

El sistema **MPNet + flan-t5-base + FAISS** resulta funcional para atención al cliente bancario. Las principales limitaciones son:

- `flan-t5-base` no está *fine-tuneado* en español bancario → respuestas pueden ser genéricas o en inglés en casos edge.
- MPNet, aunque superior a MiniLM, podría mejorarse con un modelo de embedding específico del dominio.
- Un LLM de mayor tamaño (7B+ con cuantización) o fine-tuneado en español mejoraría significativamente la Answer Relevancy y la calidad de la generación.

---
## g) Referencias

> **Consigna:** Es obligatorio citar las fuentes de todo material utilizado para su TP incluido chats con IA generativa. Al final en su apartado Referencias liste todas las fuentes consultadas.

---

1. **Google / Flan-T5 – HuggingFace Model Card:** https://huggingface.co/google/flan-t5-base  
2. **Sentence Transformers – MiniLM multilingual:** https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2  
3. **Sentence Transformers – MPNet multilingual:** https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2  
4. **FAISS – Facebook AI Similarity Search:** https://github.com/facebookresearch/faiss  
5. **Sentence Transformers – Documentación oficial:** https://www.sbert.net/  
6. **Lewis et al. (2020) – Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks:** https://arxiv.org/abs/2005.11401  
7. **Esš et al. (2021) – BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of IR:** https://arxiv.org/abs/2104.08663  
8. **RAGAS – Evaluation Framework for RAG pipelines:** https://docs.ragas.io/  
9. **HuggingFace Transformers – Documentación:** https://huggingface.co/docs/transformers  
10. **Conversación con Claude (Anthropic) –** Asistencia en el diseño e implementación del notebook TP5, junio 2026.